In [ ]:

#@title Install

from IPython.display import HTML, display

!pip install -U yt-dlp -q
!pip install moviepy ffmpeg-progress-yield tqdm -q
!apt-get update -qq
!apt-get install -y ffmpeg -qq
!wget -q https://github.com/rastikerdar/vazir-font/releases/download/v30.1.0/Vazir-font-v30.1.0.zip
!unzip -qo Vazir-font-v30.1.0.zip -d /usr/share/fonts/truetype/vazir

display(HTML("""
<p style="font-family: Vazir, Arial, sans-serif; font-size: 16px; text-align: center; color: #ffcc00; margin-bottom: 8px;">
شاید این ویدیو هم مورد علاقتون باشه 😊
</p>
<iframe width="280" height="157" src="https://www.youtube-nocookie.com/embed/ds4PsMYEj18"
title="ویدیوی پیشنهادی" frameborder="0"
allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture"
allowfullscreen
referrerpolicy="strict-origin-when-cross-origin"></iframe>
"""))

In [ ]:
#@title Connect To Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:

#@title Select Video

import os
import yt_dlp
from google.colab import files

# انتخاب روش دریافت ویدیو
upload_method = "Upload from local storage" #@param ["Upload from local storage", "Enter Drive path", "Download from YouTube"]

# اگر از درایو یا یوتیوب باشه، اینجا لینک یا مسیر رو وارد کن
source_path = "" #@param {type:"string"}

def download_from_youtube(url):
    print("در حال دانلود ویدیو از یوتیوب...")

    ydl_opts = {
        'format': 'bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best',  # اولویت با mp4 مستقیم
        'outtmpl': 'youtube_video.%(ext)s',
        'merge_output_format': 'mp4',                   # حتماً خروجی mp4 بشه
        'concurrent_fragment_downloads': 5,             # سرعت دانلود بالاتر
        'retries': 10,
        'fragment_retries': 10,
        'no_warnings': False,
        'ignoreerrors': False,
    }

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(url, download=True)
            filename = ydl.prepare_filename(info)

        # اگر فایل با اکستنشن دیگه‌ای دانلود شده باشه، به mp4 تغییر نام بده
        if not os.path.exists('youtube_video.mp4'):
            if os.path.exists(filename):
                os.rename(filename, 'youtube_video.mp4')
                print("فایل دانلود شده به youtube_video.mp4 تغییر نام داده شد.")
            else:
                print("خطا: فایل دانلود شده پیدا نشد!")
                return None

        print("✓ دانلود ویدیو با موفقیت انجام شد!")
        return os.path.join(os.getcwd(), 'youtube_video.mp4')

    except Exception as e:
        print("❌ خطا در دانلود از یوتیوب:")
        print(str(e))
        print("\nپیشنهاد: ویدیو رو روی کامپیوتر خودتون با yt-dlp یا سایت‌هایی مثل ssyoutube.com دانلود کنید و بعد از طریق 'Upload from local storage' آپلود کنید.")
        return None

def get_video():
    if upload_method == "Upload from local storage":
        print("لطفاً فایل ویدیویی را آپلود کنید...")
        uploaded = files.upload()
        if not uploaded:
            print("هیچ فایلی آپلود نشد!")
            return None
        video_path = list(uploaded.keys())[0]
        return os.path.join(os.getcwd(), video_path)

    elif upload_method == "Enter Drive path":
        if not source_path.strip():
            print("لطفاً مسیر فایل در گوگل درایو را وارد کنید!")
            return None
        if not os.path.exists(source_path):
            print(f"فایل پیدا نشد: {source_path}")
            print("مطمئن شوید مسیر درست باشد و درایو مانت شده باشد.")
            return None
        return source_path

    else:  # Download from YouTube
        if not source_path.strip():
            print("لطفاً لینک یوتیوب را وارد کنید!")
            return None
        return download_from_youtube(source_path)

# اجرای دریافت ویدیو
video_path = get_video()

if video_path and os.path.exists(video_path):
    print(f"✓ ویدیو با موفقیت بارگذاری شد:")
    print(f"   مسیر: {video_path}")
else:
    print("❌ مشکلی در بارگذاری ویدیو وجود دارد. لطفاً دوباره امتحان کنید.")

In [ ]:
#@title Upload Subtitle
from google.colab import files
import os


if os.path.exists('loren.srt'):
  os.remove('loren.srt')

print("لطفاً فایل زیرنویس (SRT) خود را آپلود کنید:")
uploaded = files.upload()

if uploaded:

    original_filename = next(iter(uploaded))


    os.rename(original_filename, 'loren.srt')

    subtitle_path = '/content/loren.srt'
    print(f"✅ فایل '{original_filename}' با موفقیت دریافت شد.")
    print("حالا می‌توانید سلول Run را اجرا کنید.")
else:
    print("❌ هیچ فایلی آپلود نشد.")

In [ ]:

#@title Run
import os
from ffmpeg_progress_yield import FfmpegProgress
from tqdm.notebook import tqdm


video_path = '/content/youtube_video.mp4'
subtitle_path = '/content/loren.srt'
background_style = "بک‌گراند کم‌رنگ (نیمه‌شفاف)" #@param ["بدون بک‌گراند", "بک‌گراند کم‌رنگ (نیمه‌شفاف)", "بک‌گراند غلیظ (مشکی کامل)"]

def add_subtitle_ffmpeg(video_path, subtitle_path, output_path):

    if not os.path.exists(subtitle_path):
        print(f"❌ خطا: فایل زیرنویس در مسیر {subtitle_path} پیدا نشد!")
        print("لطفاً از پنل سمت چپ مطمئن شوید نام فایل دقیقاً همین است.")
        return


    style_params = [
        'Fontname=Vazir', 'Fontsize=24', 'Bold=1',
        'PrimaryColour=&H00FFFF', 'Alignment=2', 'MarginV=30'
    ]

    if background_style == "بدون بک‌گراند":
        style_params.extend(['BorderStyle=1', 'Outline=2', 'Shadow=1', 'BackColour=&H00000000'])
    elif background_style == "بک‌گراند کم‌رنگ (نیمه‌شفاف)":
        style_params.extend(['BorderStyle=4', 'BackColour=&H80000000', 'Outline=0'])
    else:
        style_params.extend(['BorderStyle=4', 'BackColour=&H00000000', 'Outline=0'])

    style_string = ','.join(style_params)


    filter_cmd = f"subtitles={subtitle_path}:force_style='{style_string}'"

    command = [
        'ffmpeg', '-i', video_path,
        '-vf', filter_cmd,
        '-c:v', 'h264_nvenc',
        '-preset', 'fast',
        '-b:v', '5M',
        '-c:a', 'copy',
        '-y', output_path
    ]

    print(f"🎬 شروع هاردساب...")

    try:
        ff = FfmpegProgress(command)
        with tqdm(total=100, desc="Processing", unit="%") as pbar:
            for progress in ff.run_command_with_progress():
                pbar.n = progress
                pbar.refresh()
        print("✓ زیرنویس با موفقیت چسبانده شد!")
    except Exception as e:
        print(f"❌ خطای سیستم: {e}")

# اجرا
output_path = '/content/output.mp4'
add_subtitle_ffmpeg(video_path, subtitle_path, output_path)

In [ ]:
#@title Download large file
#@markdown ####ذخیره در درایو با حجم واقعی
from google.colab import drive
import os
import shutil
from datetime import datetime

# Mount Google Drive
drive.mount('/content/drive')

# Create folder
save_path = '/content/drive/My Drive/Subtitle_Videos'
os.makedirs(save_path, exist_ok=True)

# Check and copy all possible output files
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_files = {
    '/content/output.mp4': f'normal_video_{timestamp}.mp4',
    '/content/shorts_output.mp4': f'shorts_video_{timestamp}.mp4'
}

for source, dest_name in output_files.items():
    if os.path.exists(source):
        drive_path = os.path.join(save_path, dest_name)
        shutil.copy(source, drive_path)
        print(f"✓ Saved: {drive_path}")

In [ ]:
#@title Download compressed
#@markdown ####ذخیره در درایو با حجم کمتر
from google.colab import drive
import os
import shutil
import subprocess
from datetime import datetime

# Mount Drive
drive.mount('/content/drive')

# Create folder
save_path = '/content/drive/My Drive/Subtitle_Videos'
os.makedirs(save_path, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Check which output file exists
if os.path.exists('/content/output.mp4'):
    source_file = '/content/output.mp4'
    dest_name = f'compressed_{timestamp}.mp4'
elif os.path.exists('/content/shorts_output.mp4'):
    source_file = '/content/shorts_output.mp4'
    dest_name = f'compressed_shorts_{timestamp}.mp4'
else:
    print("⚠️ No output video found!")
    exit()

# Create compressed version
compressed_output = f'/content/compressed_{timestamp}.mp4'
compress_command = [
    'ffmpeg', '-i', source_file,
    '-c:v', 'h264_nvenc',
    '-preset', 'fast',
    '-crf', '23',
    '-c:a', 'aac',
    '-b:a', '128k',
    compressed_output
]
subprocess.run(compress_command)

# Save to Drive
drive_path = os.path.join(save_path, dest_name)
shutil.copy(compressed_output, drive_path)
print(f"✓ Compressed video saved successfully!")
print(f"Full path: {drive_path}")

In [ ]:

#@title 📥 Download Output
#@markdown دانلود خودکار در حافظه داخلی
import os
from google.colab import files


output_file = '/content/output.mp4'

if os.path.exists(output_file):
    print(f"در حال آماده‌سازی فایل {output_file} برای دانلود...")
    files.download(output_file)
else:
    print("❌ خطای دانلود: فایل خروجی پیدا نشد. ابتدا سلول Run را اجرا کنید.")